# Palace mesh: two-qubit chip

Mesh the two-qubit, straight-bus variant of QPDK's
[Filled Qubit Test Chip](https://github.com/gdsfactory/quantum-rf-pdk/blob/main/qpdk/samples/filled_test_chip.py).
See the PDK for layout details; the variant is defined in
[this YAML](data/qpdk/two_qubit_straight_bus.pic.yml).

Use the repo's Python kernel with `qpdk==0.4.0` installed. This notebook
generates a local mesh only: planar PEC metal on silicon, with vacuum above.
Nanoscale junction layers are omitted. Lengths are in µm.

In [ ]:
from pathlib import Path
from time import perf_counter

import gdsfactory as gf
import klayout.db as kdb
from qpdk import PDK
from qpdk.samples.filled_test_chip import filled_qubit_test_chip
from qpdk.tech import LAYER, LAYER_STACK, material_properties

from gsim.common.polygon_utils import simplify_polygon
from gsim.common.stack import Layer, LayerStack
from gsim.palace import EigenmodeSim

PDK.activate()

## Settings

`10.0` is coarse; try `2.0` or `1.0` for the smallest 2 µm features.
Each refinement writes to its own output folder.

In [ ]:
refined_mesh_size_um = 10.0
max_mesh_size_um = 300.0
curve_tolerance_um = 0.1  # Set 0 to preserve every mask vertex.
interactive_mesh = False

notebook_dir = Path.cwd() / "nbs" if (Path.cwd() / "nbs").is_dir() else Path.cwd()
output_dir = (
    notebook_dir
    / "palace-sim-qpdk-two-qubit-straight-bus"
    / f"mesh-{refined_mesh_size_um:g}um"
)
output_dir.mkdir(parents=True, exist_ok=True)

## Measured mesh sizes

Local runs with `max_mesh_size_um=300` and `curve_tolerance_um=0.1`.
All meshes passed validation with zero invalid elements. Times exclude previews
and depend on hardware; file sizes use decimal MB.

| Refinement (µm) | Nodes | Tetrahedra | Mesh file (MB) | Time (min) |
| ---: | ---: | ---: | ---: | ---: |
| 10 | 96,438 | 520,700 | 28.7 | 1.0 |
| 5 | 181,733 | 995,617 | 57.8 | 1.6 |
| 2 | 487,869 | 2,682,411 | 161.9 | 3.7 |
| 1 | 955,579 | 5,226,904 | 319.9 | 8.1 |

## Layout and metal mask

Load the PDK example, then subtract its etch mask from the chip footprint.

In [ ]:
chip = filled_qubit_test_chip(
    yaml_path=notebook_dir / "data/qpdk/two_qubit_straight_bus.pic.yml"
)
chip.write_gds(output_dir / "layout.gds")
chip.plot()

In [ ]:
layout = chip.kdb_cell.layout()
etch = kdb.Region(chip.kdb_cell.begin_shapes_rec(layout.layer(*LAYER.M1_ETCH))).merged()
assert not etch.is_empty(), "Missing M1 etch mask."
footprint = kdb.Region(etch.bbox())
original_metal = footprint - etch
if curve_tolerance_um > 0:
    etch = kdb.Region(
        [simplify_polygon(p, tolerance=curve_tolerance_um) for p in etch.each()]
    ).merged()
metal = footprint - etch
assert metal.count() == original_metal.count()
assert sum(p.holes() for p in metal.each()) == sum(
    p.holes() for p in original_metal.each()
)

mesh_layers = {"M1": (201, 0), "SUBSTRATE": (202, 0), "VACUUM": (203, 0)}
mesh_component = gf.Component()
for name, region in (("M1", metal), ("SUBSTRATE", footprint), ("VACUUM", footprint)):
    mesh_component.kdb_cell.shapes(
        mesh_component.kcl.layout.layer(*mesh_layers[name])
    ).insert(region)

## Stack

Use QPDK's substrate and vacuum thicknesses, with the PEC sheet at `z=0`.

In [ ]:
stack = LayerStack(pdk_name="qpdk")
for name, zmin, zmax, material, layer_type in (
    (
        "SUBSTRATE",
        -LAYER_STACK.layers["Substrate"].thickness,
        0.0,
        "silicon",
        "dielectric",
    ),
    ("M1", 0.0, 0.0, "niobium", "conductor"),
    ("VACUUM", 0.0, LAYER_STACK.layers["Vacuum"].thickness, "vacuum", "dielectric"),
):
    stack.layers[name] = Layer(
        name=name,
        gds_layer=mesh_layers[name],
        zmin=zmin,
        zmax=zmax,
        thickness=zmax - zmin,
        material=material,
        layer_type=layer_type,
    )
stack.dielectrics = [
    dict(
        name=layer.name.lower(),
        zmin=layer.zmin,
        zmax=layer.zmax,
        material=layer.material,
    )
    for layer in stack.layers.values()
    if layer.layer_type == "dielectric"
]
stack.materials = {
    "silicon": {
        "type": "dielectric",
        "permittivity": material_properties["Si"]["relative_permittivity"],
        "loss_tangent": material_properties["Si"]["loss_tangent"],
    },
    "vacuum": {"type": "dielectric", "permittivity": 1.0},
    "niobium": {"type": "conductor"},
}

sim = EigenmodeSim()
sim.set_geometry(mesh_component)
sim.set_stack(stack)
sim.set_output_dir(output_dir)
# mesh() validates the eigenmode setup, which needs a target frequency.
sim.set_eigenmode(num_modes=1, target=5e9)

## Mesh and preview

In [ ]:
started = perf_counter()
mesh_result = sim.mesh(
    preset="coarse",
    refined_mesh_size=refined_mesh_size_um,
    max_mesh_size=max_mesh_size_um,
    fmax=10e9,
    planar_conductors=True,
    auto_size=False,
)
elapsed_seconds = perf_counter() - started
assert sim.validate_mesh().valid
assert mesh_result.mesh_stats["sicn"]["invalid"] == 0
print(mesh_result)
print(
    f"Time: {elapsed_seconds / 60:.1f} min; file: {mesh_result.mesh_path.stat().st_size / 1e6:.1f} MB"
)

In [ ]:
sim.plot_mesh(
    output=output_dir / "mesh_metal.png",
    show_groups=["M1"],
    interactive=interactive_mesh,
)